In [1]:
import sys
print(sys.executable)

/lustre/orion/gen006/scratch/sireesiru/miniffinity/conda_env/bin/python


In [2]:
import torch

In [7]:
! pip install --upgrade --quiet bitsandbytes datasets peft transformers trl
!huggingface-cli login
## run these 2 commands on terminal## get your token from Huggingface and set it. Also set it as an envir variable 

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): Traceb

In [ ]:
##### Load TxGemma models form Huggingface 
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/txgemma-2b-predict"

# Use 4-bit quantization to reduce memory usage
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map={"":0},
    torch_dtype="auto",
    attn_implementation="eager",
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

In [9]:
import os
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------
DATASET_DIR = "/lustre/orion/gen006/scratch/sireesiru/miniffinity/datasets"
MODEL_ID = "google/txgemma-2b-predict"   # requires Hugging Face authentication
SAVE_DIR = os.path.join(DATASET_DIR, "embeddings")
os.makedirs(SAVE_DIR, exist_ok=True)

# ---------------------------------------------------------------------
# LOAD MODEL AND TOKENIZER
# ---------------------------------------------------------------------
# Make sure you have authenticated via: huggingface-cli login
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_auth_token=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto")
model.eval()

print(f"Model {MODEL_ID} loaded successfully.")

# ---------------------------------------------------------------------
# FUNCTION: generate embeddings for one dataset
# ---------------------------------------------------------------------
def generate_embeddings_for_dataset(csv_path, text_column=None):
    print(f"\nProcessing dataset: {csv_path}")
    df = pd.read_csv(csv_path)
    
    # Automatically detect the likely text/SMILES column if not specified
    if text_column is None:
        candidates = [c for c in df.columns if "smile" in c.lower() or "text" in c.lower() or "seq" in c.lower()]
        text_column = candidates[0] if candidates else df.columns[0]
        print(f" → Using column '{text_column}' for embedding generation")

    embeddings = []
    items = df[text_column].astype(str).tolist()

    for i, item in enumerate(items):
        inputs = tokenizer(item, return_tensors="pt", truncation=True, max_length=256).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            last_hidden_state = outputs.hidden_states[-1]
            pooled_embedding = last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()
            embeddings.append(pooled_embedding)

        if (i + 1) % 50 == 0:
            print(f"   → Processed {i+1}/{len(items)} entries")

    embeddings_np = np.vstack(embeddings)
    df_embeddings = pd.DataFrame(embeddings_np)
    df_embeddings.insert(0, text_column, items)

    # Save embeddings as .csv and .npy
    base_name = os.path.splitext(os.path.basename(csv_path))[0]
    csv_out = os.path.join(SAVE_DIR, f"{base_name}_embeddings.csv")
    npy_out = os.path.join(SAVE_DIR, f"{base_name}_embeddings.npy")

    df_embeddings.to_csv(csv_out, index=False)
    np.save(npy_out, embeddings_np)

    print(f"✓ Saved embeddings for {base_name} →")
    print(f"   {csv_out}")
    print(f"   {npy_out}")

# ---------------------------------------------------------------------
# LOOP THROUGH ALL CSV FILES IN YOUR DATASET FOLDER
# ---------------------------------------------------------------------
csv_files = [f for f in os.listdir(DATASET_DIR) if f.endswith(".csv")]
if not csv_files:
    print(f"No CSV files found in {DATASET_DIR}. Please verify your path.")
else:
    for csv_file in csv_files:
        csv_path = os.path.join(DATASET_DIR, csv_file)
        generate_embeddings_for_dataset(csv_path)

print("\nAll datasets processed successfully!")

/lustre/orion/gen006/scratch/sireesiru/miniffinity/conda_env/lib/python3.12/site-packages/transformers/models/auto/tokenization_auto.py:1025: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model google/txgemma-2b-predict loaded successfully.

Processing dataset: /lustre/orion/gen006/scratch/sireesiru/miniffinity/datasets/cache_1.csv
 → Using column 'smiles' for embedding generation
   → Processed 50/2534 entries
   → Processed 100/2534 entries
   → Processed 150/2534 entries
   → Processed 200/2534 entries
   → Processed 250/2534 entries
   → Processed 300/2534 entries
   → Processed 350/2534 entries
   → Processed 400/2534 entries
   → Processed 450/2534 entries
   → Processed 500/2534 entries
   → Processed 550/2534 entries
   → Processed 600/2534 entries
   → Processed 650/2534 entries
   → Processed 700/2534 entries
   → Processed 750/2534 entries
   → Processed 800/2534 entries
   → Processed 850/2534 entries
   → Processed 900/2534 entries
   → Processed 950/2534 entries
   → Processed 1000/2534 entries
   → Processed 1050/2534 entries
   → Processed 1100/2534 entries
   → Processed 1150/2534 entries
   → Processed 1200/2534 entries
   → Processed 1250/2534 entries